In [1]:
# Imports

import helpers.helper_functions as hf
import mne
import matplotlib.pyplot as plt
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path



ss = hf.settings_dict()

In [2]:
# Parameters

retina_channels = ['EOG003', 'EOG004']

#data covariance times
tmax = 4.5
tmin = 0.01
method = 'empirical'

#beamformer parameters
reg=0.05
pick_ori='max-power'
weight_norm='nai'

In [3]:


for subject_index in ss['subject_idx_list']:

    subject = ss['subject_id_list'][subject_index]
    print("loading raw dataset for subject: ", subject)

    epochs = mne.read_epochs(op.join(ss['epochs_dir'],subject+"-epo.fif"), preload=True)

    # loop over each event type
    for event_name in epochs.event_id.keys():

        save_dir = Path(ss['stc_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        #average
        evoked = epochs[event_name].copy().average()

        # extract the retina signal and save it to a file
        times = evoked.times

        if len(retina_channels) > 0:
            # Convert names to indices
            retina_indices = [evoked.ch_names.index(ch) for ch in retina_channels]

            # Create a new Evoked with the averaged channel
            retina = combine_channels(
                evoked,
                groups=dict(avg=retina_indices),
                method='mean'
            )

            retina_data = retina.data[0]
        else:

            idx = evoked.ch_names.index(retina_channels[0])
            # Extract data

            retina_data = evoked.data[idx]

        # Create dataframe
        df = pd.DataFrame({
            "time_s": times,
            "amplitude": retina_data,
        })

        # Save to CSV
        df.to_csv(save_dir / f"{subject}-event-{event_name}-retina.csv" , index=False)

        # now manking the stc
        evoked.pick('grad')

        #Load forward model
        #TODO: check that the forward models are correct
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        # some of the forward models have the wrong subject name
        if fwd['src'][0]['subject_his_id'].startswith("fs"):
            fwd['src'][0]['subject_his_id'] = subject

        #Source localization

        #make data covariance matrix
        #TODO: check that the data covariance parameters are correct
        data_cov = mne.compute_covariance(epochs, tmin=tmin, tmax=tmax, method=method)

        sp_filter = make_lcmv(
            evoked.info,
            fwd,
            data_cov=data_cov,
            reg=reg,
            pick_ori=pick_ori,
            weight_norm=weight_norm,
            rank=None,
        )

        stc = mne.beamformer.apply_lcmv(evoked, sp_filter)

        stc.save(save_dir / f"{subject}-event-{event_name}-vol.stc" , overwrite=True)

    del epochs

loading raw dataset for subject:  0005_3SJ
Reading /media/elias/Personal Data/Documents/masters/Thesis/data/scratch/epochs/0005_3SJ-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    4500.00 ms
        0 CTF compensation matrices available
Not setting metadata
70 matching events found
No baseline correction applied
0 projection items activated
Applying baseline correction (mode: mean)
Reading forward solution from /media/elias/Personal Data/Documents/masters/Thesis/data/scratch/fwd/0005_3SJ-fwd.fif...
    Reading a source space...
    [done]
    1 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read MEG forward solution (1440 sources, 306 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Reducing data rank from 208 -> 208
Estimating covariance using EMPIRICAL
Done.
Number of samples used : 628670
[done]
Computing rank from covariance with rank='info'
   